# RAG Image Caption Indexing + Synthetic Data Generation

This notebook builds an end-to-end pipeline using **OpenAI-compatible clients**:

- A **vision model** to caption each image.
- A **text embedding model** to embed captions.
- A **vector DB (ChromaDB)** to store and search image-caption vectors.
- An optional **large text model** to generate synthetic fine-tuning data from books and retrieved context.

The notebook is designed to run top-to-bottom and includes retry logic, batching, and checkpoints.

In [ ]:
# Uncomment if running on a fresh environment
# %pip install -q openai chromadb pillow tqdm python-dotenv pymupdf

/bin/bash: /home/sasank-v/anaconda3/envs/ml/lib/libtinfo.so.6: no version information available (required by /bin/bash)


## 1. Set Up OpenAI-Compatible Clients and Environment

Install dependencies, import libraries, load API keys from environment variables, and configure two OpenAI-compatible endpoints:

- `vision_client`: image-to-text generation
- `text_client`: text embeddings and synthetic data generation

In [1]:
import base64
import json
import logging
import os
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Tuple

import chromadb
import fitz
from dotenv import load_dotenv
from openai import OpenAI
from tqdm.auto import tqdm

load_dotenv()

# Update these environment variables in your shell/.env
# VISION_BASE_URL, VISION_API_KEY, VISION_MODEL
# TEXT_BASE_URL, TEXT_API_KEY, SYNTHETIC_MODEL (optional)
# EMBEDDING_BASE_URL, EMBEDDING_API_KEY, EMBEDDING_MODEL

VISION_BASE_URL = os.getenv("VISION_BASE_URL", "https://vision-llm.ipropel.co.in/v1")
VISION_API_KEY = os.getenv("VISION_API_KEY", "")
VISION_MODEL = os.getenv("VISION_MODEL", "Qwen/Qwen3-VL-8B-Instruct")

TEXT_BASE_URL = os.getenv("TEXT_BASE_URL", "https://llm.ipropel.co.in/v1")
TEXT_API_KEY = os.getenv("TEXT_API_KEY", "")
SYNTHETIC_MODEL = os.getenv("SYNTHETIC_MODEL", "openai/gpt-oss-20b")

EMBEDDING_BASE_URL = os.getenv("EMBEDDING_BASE_URL", "https://embedding-llm.ipropel.co.in/v1")
EMBEDDING_API_KEY = os.getenv("EMBEDDING_API_KEY", "")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "Qwen/Qwen3-Embedding-0.6B")

vision_client = OpenAI(base_url=VISION_BASE_URL, api_key=VISION_API_KEY)
text_client = OpenAI(base_url=TEXT_BASE_URL, api_key=TEXT_API_KEY)
embedding_client = OpenAI(base_url=EMBEDDING_BASE_URL, api_key=EMBEDDING_API_KEY)

# Notebook-local paths
ROOT = Path.cwd().resolve()
BOOKS_DIR = ROOT.parents[1] / "datasets" / "training" / "Books"
RAW_IMAGE_DIR = ROOT / "images"
PDF_IMAGE_DIR = RAW_IMAGE_DIR / "books"
IMAGE_DIR = PDF_IMAGE_DIR
VECTORDB_DIR = ROOT / "vectordb"
CHECKPOINT_DIR = ROOT / "checkpoints"
OUTPUT_DIR = ROOT / "outputs"

RAW_IMAGE_DIR.mkdir(parents=True, exist_ok=True)
PDF_IMAGE_DIR.mkdir(parents=True, exist_ok=True)
VECTORDB_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("BOOKS_DIR:", BOOKS_DIR)
print("IMAGE_DIR (extracted):", IMAGE_DIR)
print("VECTORDB_DIR:", VECTORDB_DIR)
print("EMBEDDING_BASE_URL:", EMBEDDING_BASE_URL)

BOOKS_DIR: /media/sasank-v/New Volume/Studies/College/Interships/LLM-Finetuning/datasets/training/Books
IMAGE_DIR (extracted): /media/sasank-v/New Volume/Studies/College/Interships/LLM-Finetuning/notebooks/preprocessing/images/books
VECTORDB_DIR: /media/sasank-v/New Volume/Studies/College/Interships/LLM-Finetuning/notebooks/preprocessing/vectordb
EMBEDDING_BASE_URL: https://embedding-llm.ipropel.co.in/v1


## Extract Images from Book PDFs

This follows the same approach as your `data-cleaning.ipynb`: iterate over each PDF in the Books folder, extract page images, and keep only images above a minimum size threshold.

In [2]:
def extract_images_from_pdf(pdf_path: Path, output_folder: Path, min_width: int = 250, min_height: int = 250) -> int:
    output_folder.mkdir(parents=True, exist_ok=True)
    doc = fitz.open(pdf_path)
    extracted = 0

    for page_number, page in enumerate(doc, start=1):
        image_list = page.get_images(full=True)
        for img_index, img in enumerate(image_list, start=1):
            xref = img[0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]
            image_ext = base_image.get("ext", "png")
            width = base_image.get("width", 0)
            height = base_image.get("height", 0)

            if width < min_width or height < min_height:
                continue

            image_filename = output_folder / f"{pdf_path.stem}_page{page_number}_img{img_index}.{image_ext}"

            if image_filename.exists():
                continue

            with open(image_filename, "wb") as img_file:
                img_file.write(image_bytes)
            extracted += 1

    doc.close()
    return extracted


def extract_images_from_books(books_dir: Path, output_folder: Path, min_width: int = 250, min_height: int = 250) -> int:
    pdf_files = sorted(books_dir.glob("*.pdf"))
    total = 0

    for pdf_path in tqdm(pdf_files, desc="Extracting images from books"):
        try:
            n = extract_images_from_pdf(
                pdf_path=pdf_path,
                output_folder=output_folder,
                min_width=min_width,
                min_height=min_height,
            )
            total += n
        except Exception as exc:
            print(f"Failed for {pdf_path.name}: {exc}")

    return total


MIN_IMAGE_WIDTH = int(os.getenv("MIN_IMAGE_WIDTH", "250"))
MIN_IMAGE_HEIGHT = int(os.getenv("MIN_IMAGE_HEIGHT", "250"))

if not BOOKS_DIR.exists():
    raise FileNotFoundError(f"Books directory not found: {BOOKS_DIR}")

new_images = extract_images_from_books(
    books_dir=BOOKS_DIR,
    output_folder=IMAGE_DIR,
    min_width=MIN_IMAGE_WIDTH,
    min_height=MIN_IMAGE_HEIGHT,
)

print(f"Newly extracted images: {new_images}")
print(f"All extracted images path: {IMAGE_DIR}")

Extracting images from books:   0%|          | 0/12 [00:00<?, ?it/s]

Newly extracted images: 0
All extracted images path: /media/sasank-v/New Volume/Studies/College/Interships/LLM-Finetuning/notebooks/preprocessing/images/books


## 2. Initialize Vector Database Collection

Connect to ChromaDB, create/select a collection, and define metadata fields for:

- image path
- generated caption
- vision model
- embedding model
- timestamps

In [3]:
client = chromadb.PersistentClient(path=str(VECTORDB_DIR))
COLLECTION_NAME = "image_caption_index"

collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={
        "description": "Image captions embedded with OpenAI-compatible text model",
        "hnsw:space": "cosine",
    },
)

print("Collection:", collection.name)

Collection: image_caption_index


## 3. Load and Normalize Image Inputs

Read image files from the target folder, validate formats, normalize paths/IDs, and create a dataset for processing.

In [4]:
SUPPORTED_EXTS = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}


def list_images(image_dir: Path) -> List[Path]:
    all_files = sorted([p for p in image_dir.rglob("*") if p.is_file()])
    images = [p for p in all_files if p.suffix.lower() in SUPPORTED_EXTS]
    return images


image_files = list_images(IMAGE_DIR)
image_items: List[Dict[str, Any]] = []

for p in image_files:
    rel = p.relative_to(ROOT)
    image_items.append(
        {
            "id": f"img::{rel.as_posix()}",
            "path": p,
            "relative_path": rel.as_posix(),
        }
    )

print(f"Found {len(image_items)} images")
if image_items:
    print("Sample:", image_items[0]["relative_path"])

Found 969 images
Sample: images/books/CAO - 1_page103_img1.jpeg


## 4. Generate Captions from Images with the Vision Model

For each image, call the vision model, parse the response safely, and keep intermediate outputs in memory/checkpoints.

In [6]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("rag-image-pipeline")

CAPTION_CHECKPOINT = CHECKPOINT_DIR / "captions.json"

# Keep prompts short to stay under strict vision model context limits.
SYSTEM_MESSAGE = (
    "You are a Computer Science technical writer. "
    "Write concise, factual captions for textbook figures."
)

USER_PROMPT_LONG = (
    "Describe the key CS concept, major components, and important relationships "
    "in 2-4 sentences using technical terminology."
)

USER_PROMPT_SHORT = (
    "Give a concise 1-2 sentence technical caption of this CS figure."
)


def get_image_mime(image_path: Path) -> str:
    ext = image_path.suffix.lower()
    if ext == ".png":
        return "image/png"
    if ext in {".jpg", ".jpeg"}:
        return "image/jpeg"
    if ext == ".webp":
        return "image/webp"
    if ext == ".bmp":
        return "image/bmp"
    return "image/jpeg"


def prepare_image_base64(image_path: Path, max_side: int = 1024, jpeg_quality: int = 85) -> tuple[str, str]:
    """Resize large images before sending to reduce multimodal token usage."""
    from io import BytesIO
    from PIL import Image

    with Image.open(image_path) as img:
        img = img.convert("RGB")
        w, h = img.size
        if max(w, h) > max_side:
            img.thumbnail((max_side, max_side), Image.Resampling.LANCZOS)

        buf = BytesIO()
        img.save(buf, format="JPEG", quality=jpeg_quality, optimize=True)
        image_bytes = buf.getvalue()

    image_base64 = base64.b64encode(image_bytes).decode("utf-8")
    return image_base64, "image/jpeg"


def is_prompt_too_long_error(exc: Exception) -> bool:
    text = str(exc).lower()
    return (
        "maximum model length" in text
        or "decoder prompt" in text
        or "prompt" in text and "longer than" in text
    )


def with_retry(func, retries: int = 5, base_sleep: float = 1.5):
    for attempt in range(1, retries + 1):
        try:
            return func()
        except Exception as exc:
            # Deterministic context-limit errors should not be retried blindly.
            if is_prompt_too_long_error(exc):
                raise
            if attempt == retries:
                raise
            sleep_s = base_sleep * (2 ** (attempt - 1))
            logger.warning("Attempt %s failed: %s. Retrying in %.1fs", attempt, exc, sleep_s)
            time.sleep(sleep_s)


def _caption_call(image_base64: str, image_mime: str, user_prompt: str) -> str:
    resp = vision_client.chat.completions.create(
        model=VISION_MODEL,
        temperature=0.1,
        max_tokens=220,
        messages=[
            {"role": "system", "content": SYSTEM_MESSAGE},
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": user_prompt},
                    {"type": "image_url", "image_url": {"url": f"data:{image_mime};base64,{image_base64}"}},
                ],
            },
        ],
    )
    return (resp.choices[0].message.content or "").strip()


def caption_image(image_path: Path) -> str:
    image_base64, image_mime = prepare_image_base64(image_path, max_side=1024)

    try:
        return with_retry(lambda: _caption_call(image_base64, image_mime, USER_PROMPT_LONG))
    except Exception as exc:
        if not is_prompt_too_long_error(exc):
            raise

    # Fallback path for strict context limits: downscale harder + shorter prompt.
    logger.warning("Context limit hit for %s. Retrying with compact image/prompt.", image_path.name)
    image_base64_small, image_mime_small = prepare_image_base64(image_path, max_side=768, jpeg_quality=75)
    return _caption_call(image_base64_small, image_mime_small, USER_PROMPT_SHORT)


def load_json(path: Path, default: Any):
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))
    return default


def dump_json(path: Path, payload: Any):
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")


captions_state = load_json(CAPTION_CHECKPOINT, default={})

for item in tqdm(image_items, desc="Captioning images"):
    if item["id"] in captions_state:
        continue

    try:
        caption = caption_image(item["path"])
        captions_state[item["id"]] = {
            "caption": caption,
            "relative_path": item["relative_path"],
            "vision_model": VISION_MODEL,
            "captioned_at": datetime.now(timezone.utc).isoformat(),
        }
        dump_json(CAPTION_CHECKPOINT, captions_state)
    except Exception as exc:
        logger.error("Caption failed for %s: %s", item["relative_path"], exc)

print(f"Captured captions for {len(captions_state)} images")
if captions_state:
    sample_key = next(iter(captions_state))
    print(sample_key, "->", captions_state[sample_key]["caption"][:180])

Captioning images:   0%|          | 0/969 [00:00<?, ?it/s]

Captured captions for 969 images
img::images/books/CAO - 1_page103_img1.jpeg -> This stylized diagram illustrates the architecture of a computer system, showing its core components: a Processor (containing Control and Datapath units), Memory, Input/Output inte


## 5. Create Text Embeddings from Captions

Embed each generated caption using the text embedding model and map vectors back to the image metadata.

In [7]:
EMBED_CHECKPOINT = CHECKPOINT_DIR / "caption_embeddings.json"


def embed_texts(texts: List[str]) -> List[List[float]]:
    def _call():
        resp = embedding_client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
        return [d.embedding for d in resp.data]

    return with_retry(_call)


embed_state = load_json(EMBED_CHECKPOINT, default={})

pending_ids = [k for k in captions_state.keys() if k not in embed_state]
BATCH_SIZE = 32

for i in tqdm(range(0, len(pending_ids), BATCH_SIZE), desc="Embedding captions"):
    batch_ids = pending_ids[i : i + BATCH_SIZE]
    batch_texts = [captions_state[_id]["caption"] for _id in batch_ids]
    try:
        vectors = embed_texts(batch_texts)
        for _id, vec in zip(batch_ids, vectors):
            embed_state[_id] = {
                "embedding": vec,
                "embedding_model": EMBEDDING_MODEL,
                "embedded_at": datetime.now(timezone.utc).isoformat(),
            }
        dump_json(EMBED_CHECKPOINT, embed_state)
    except Exception as exc:
        logger.error("Embedding batch failed (%s-%s): %s", i, i + len(batch_ids), exc)

print(f"Embedded captions: {len(embed_state)}")

Embedding captions: 0it [00:00, ?it/s]

Embedded captions: 969


In [8]:
import re

BOOK_TEXT_PATTERNS = ["*.txt", "*_cleaned.txt", "*_full.txt"]
CLEANED_TEXT_DIR = CHECKPOINT_DIR / "cleaned_books"
CLEANED_TEXT_DIR.mkdir(parents=True, exist_ok=True)


def find_book_text_files(books_dir: Path) -> List[Path]:
    candidates: List[Path] = []
    for pattern in BOOK_TEXT_PATTERNS:
        candidates.extend(list(books_dir.glob(pattern)))

    seen = set()
    ordered: List[Path] = []
    for p in sorted(candidates):
        if p.suffix.lower() != ".txt":
            continue
        key = str(p.resolve())
        if key in seen:
            continue
        seen.add(key)
        ordered.append(p)
    return ordered


def remove_headers_footers(text: str) -> str:
    lines = text.split("\n")
    cleaned_lines: List[str] = []

    for line in lines:
        line_strip = line.strip()

        # Remove standalone page numbers
        if re.fullmatch(r"\d+", line_strip):
            continue

        cleaned_lines.append(line)

    return "\n".join(cleaned_lines)


def remove_unwanted_sections(text: str) -> str:
    section_headings = {
        "index",
        "acknowledgment",
        "acknowledgements",
        "references",
        "bibliography",
    }

    lines = text.split("\n")
    cleaned_lines: List[str] = []
    skip = False

    for line in lines:
        line_strip = line.strip().lower()

        if line_strip in section_headings:
            skip = True
            continue

        if not skip:
            cleaned_lines.append(line)

    return "\n".join(cleaned_lines)


def fix_broken_lines(text: str) -> str:
    # Merge single newlines created by PDF/text extraction splits
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)
    # Normalize excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text


def clean_book_text(raw: str) -> str:
    text = raw
    text = remove_headers_footers(text)
    text = remove_unwanted_sections(text)
    text = fix_broken_lines(text)

    # Normalize spaces
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


# Use extracted PDF texts if available, otherwise fallback to existing txt files
if 'pdf_texts' not in globals():
    # Fallback: look for existing .txt files
    book_files = find_book_text_files(BOOKS_DIR)
    pdf_texts = {}
    for book_path in book_files:
        pdf_texts[book_path.name] = book_path.read_text(encoding="utf-8", errors="ignore")

cleaned_book_texts: Dict[str, str] = {}

for book_name, raw_text in tqdm(pdf_texts.items(), desc="Cleaning book text"):
    cleaned = clean_book_text(raw_text)
    cleaned_book_texts[book_name] = cleaned

    # Save cleaned version to disk
    out_name = book_name.replace(".pdf", "").replace(".txt", "") + "_cleaned_for_chunks.txt"
    out_path = CLEANED_TEXT_DIR / out_name
    out_path.write_text(cleaned, encoding="utf-8")

print(f"Cleaned book files: {len(cleaned_book_texts)}")
if cleaned_book_texts:
    sample_name = next(iter(cleaned_book_texts))
    print("Sample:", sample_name, "chars=", len(cleaned_book_texts[sample_name]))

Cleaning book text: 0it [00:00, ?it/s]

Cleaned book files: 0


## 5. Extract Text from Book PDFs

Before processing book text, we need to extract raw text from PDF files using PyMuPDF (fitz). This step reads all PDFs in the Books directory and creates corresponding text files.

In [9]:
def extract_text_from_pdf(pdf_path: Path, verbose: bool = True) -> str:
    """
    Extract raw text from ALL pages of a PDF file using PyMuPDF.
    Uses multiple extraction strategies to ensure comprehensive text extraction.
    """
    # Suppress MuPDF warnings for malformed PDF elements
    fitz.TOOLS.mupdf_display_errors(False)
    
    doc = fitz.open(pdf_path)
    total_pages = len(doc)
    all_text = []
    
    if verbose:
        logger.info(f"Processing {pdf_path.name}: {total_pages} pages")
    
    # Extract text from EVERY page with progress tracking
    for page_num in range(total_pages):
        page = doc[page_num]
        
        # Try multiple extraction methods for better coverage
        # Method 1: Standard text extraction
        page_text = page.get_text("text")
        
        # Method 2: If no text found, try "blocks" mode (better for complex layouts)
        if not page_text.strip():
            blocks = page.get_text("blocks")
            page_text = "\n".join([block[4] for block in blocks if len(block) > 4])
        
        # Method 3: If still no text, try extracting from text dict
        if not page_text.strip():
            text_dict = page.get_text("dict")
            page_text_parts = []
            for block in text_dict.get("blocks", []):
                if block.get("type") == 0:  # text block
                    for line in block.get("lines", []):
                        for span in line.get("spans", []):
                            page_text_parts.append(span.get("text", ""))
            page_text = " ".join(page_text_parts)
        
        all_text.append(page_text)
        
        # Log progress every 100 pages
        if verbose and (page_num + 1) % 100 == 0:
            logger.info(f"  Processed {page_num + 1}/{total_pages} pages")
    
    doc.close()
    
    combined_text = "\n".join(all_text)
    
    if verbose:
        logger.info(f"✓ {pdf_path.name}: {len(combined_text):,} chars from {total_pages} pages")
    
    return combined_text


def process_all_pdfs(books_dir: Path) -> Dict[str, str]:
    """Extract text from all PDFs in the books directory."""
    pdf_files = sorted(books_dir.glob("*.pdf"))
    extracted_texts: Dict[str, str] = {}
    
    logger.info(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_path in tqdm(pdf_files, desc="Extracting text from PDFs"):
        try:
            raw_text = extract_text_from_pdf(pdf_path, verbose=True)
            # Use PDF name as key
            extracted_texts[pdf_path.name] = raw_text
        except Exception as exc:
            logger.error(f"✗ Failed to extract text from {pdf_path.name}: {exc}")
    
    return extracted_texts


# Extract text from all PDFs
if not BOOKS_DIR.exists():
    raise FileNotFoundError(f"Books directory not found: {BOOKS_DIR}")

print("=" * 70)
print("STARTING PDF TEXT EXTRACTION")
print("=" * 70)

pdf_texts = process_all_pdfs(BOOKS_DIR)

print("\n" + "=" * 70)
print("EXTRACTION SUMMARY")
print("=" * 70)
print(f"Total PDF files processed: {len(pdf_texts)}")
print(f"\nIndividual file statistics:")
print("-" * 70)

total_chars = 0
for pdf_name, text in sorted(pdf_texts.items()):
    char_count = len(text)
    total_chars += char_count
    print(f"  {pdf_name:30s} : {char_count:>10,} chars")

print("-" * 70)
print(f"  {'TOTAL':30s} : {total_chars:>10,} chars")
print("=" * 70)

if pdf_texts:
    sample_name = next(iter(pdf_texts))
    sample_preview = pdf_texts[sample_name][:500].replace("\n", " ")
    print(f"\nSample preview from {sample_name}:")
    print(f"  {sample_preview}...")

2026-03-08 16:36:52,070 | INFO | Found 12 PDF files to process


STARTING PDF TEXT EXTRACTION


Extracting text from PDFs:   0%|          | 0/12 [00:00<?, ?it/s]

2026-03-08 16:36:52,158 | INFO | Processing CAO - 1.pdf: 1137 pages
2026-03-08 16:36:52,386 | INFO |   Processed 100/1137 pages
2026-03-08 16:36:52,551 | INFO |   Processed 200/1137 pages
2026-03-08 16:36:52,949 | INFO |   Processed 300/1137 pages
2026-03-08 16:36:53,117 | INFO |   Processed 400/1137 pages
2026-03-08 16:36:53,339 | INFO |   Processed 500/1137 pages
2026-03-08 16:36:53,496 | INFO |   Processed 600/1137 pages
2026-03-08 16:36:53,651 | INFO |   Processed 700/1137 pages
2026-03-08 16:36:53,904 | INFO |   Processed 800/1137 pages
2026-03-08 16:36:54,066 | INFO |   Processed 900/1137 pages
2026-03-08 16:36:54,239 | INFO |   Processed 1000/1137 pages
2026-03-08 16:36:54,391 | INFO |   Processed 1100/1137 pages
2026-03-08 16:36:54,471 | INFO | ✓ CAO - 1.pdf: 2,611,231 chars from 1137 pages
2026-03-08 16:36:54,480 | INFO | Processing CAO - 2.pdf: 881 pages
2026-03-08 16:36:54,583 | INFO |   Processed 100/881 pages
2026-03-08 16:36:54,677 | INFO |   Processed 200/881 pages
2026-


EXTRACTION SUMMARY
Total PDF files processed: 12

Individual file statistics:
----------------------------------------------------------------------
  CAO - 1.pdf                    :  2,611,231 chars
  CAO - 2.pdf                    :  1,919,466 chars
  CN - 1.pdf                     :  2,706,422 chars
  CN - 2.pdf                     :  2,105,879 chars
  CN - 3.pdf                     :  2,064,695 chars
  DSA - 1.pdf                    :  1,318,958 chars
  DSA - 2.pdf                    :    983,409 chars
  DSA - 3.pdf                    :  1,006,118 chars
  DSA - 4.pdf                    :  2,543,278 chars
  OS - 1.pdf                     :  3,188,670 chars
  OS - 2.pdf                     :  2,929,811 chars
  OS - 3.pdf                     :  2,022,859 chars
----------------------------------------------------------------------
  TOTAL                          : 25,400,796 chars

Sample preview from CAO - 1.pdf:
   In Praise of Computer Organization and Design: The Hardware/ Softw

## 5.5 Clean Book Text Data

Clean raw book text before chunking by removing noisy lines, normalizing whitespace, and preparing a reusable cleaned-text cache.

In [10]:
BOOK_TEXT_CHECKPOINT = CHECKPOINT_DIR / "book_text_embeddings.json"


def chunk_text(text: str, chunk_chars: int = 1400, overlap: int = 200) -> List[str]:
    if not text:
        return []
    chunks = []
    start = 0
    n = len(text)
    while start < n:
        end = min(n, start + chunk_chars)
        chunks.append(text[start:end])
        if end >= n:
            break
        start = max(0, end - overlap)
    return chunks


# Fallback in case the clean-data cell is skipped.
if "pdf_texts" not in globals():
    # Try to find existing text files
    def find_book_text_files(books_dir: Path) -> List[Path]:
        patterns = ["*.txt", "*_cleaned.txt", "*_full.txt"]
        candidates = []
        for pattern in patterns:
            candidates.extend(list(books_dir.glob(pattern)))
        seen = set()
        ordered = []
        for p in sorted(candidates):
            if p.suffix.lower() != ".txt":
                continue
            key = str(p.resolve())
            if key not in seen:
                seen.add(key)
                ordered.append(p)
        return ordered
    
    book_files = find_book_text_files(BOOKS_DIR)
    pdf_texts = {}
    for book_path in book_files:
        pdf_texts[book_path.name] = book_path.read_text(encoding="utf-8", errors="ignore")

if "cleaned_book_texts" not in globals():
    cleaned_book_texts = {}
    for book_name, raw in pdf_texts.items():
        cleaned_book_texts[book_name] = clean_book_text(raw)

print(f"Book text files found: {len(pdf_texts)}")

book_embed_state = load_json(BOOK_TEXT_CHECKPOINT, default={})
text_chunk_records: List[Dict[str, Any]] = []

for book_name in tqdm(cleaned_book_texts.keys(), desc="Processing cleaned book text"):
    file_key = book_name

    if file_key not in book_embed_state:
        cleaned = cleaned_book_texts.get(file_key, "")
        chunks = chunk_text(cleaned, chunk_chars=1400, overlap=200)

        vectors: List[List[float]] = []
        if chunks:
            for i in range(0, len(chunks), 32):
                vectors.extend(embed_texts(chunks[i : i + 32]))

        book_embed_state[file_key] = {
            "book": file_key,
            "cleaned_chars": len(cleaned),
            "chunk_count": len(chunks),
            "chunk_texts": chunks,
            "embeddings": vectors,
            "embedded_at": datetime.now(timezone.utc).isoformat(),
            "embedding_model": EMBEDDING_MODEL,
        }
        dump_json(BOOK_TEXT_CHECKPOINT, book_embed_state)

    payload = book_embed_state[file_key]
    for idx, (chunk_text_value, emb) in enumerate(zip(payload["chunk_texts"], payload["embeddings"])):
        text_chunk_records.append(
            {
                "id": f"txt::{file_key}::{idx}",
                "embedding": emb,
                "document": chunk_text_value,
                "metadata": {
                    "source_type": "book_text_chunk",
                    "book": file_key,
                    "chunk_index": idx,
                    "embedding_model": payload.get("embedding_model", EMBEDDING_MODEL),
                    "embedded_at": payload.get("embedded_at", datetime.now(timezone.utc).isoformat()),
                },
            }
        )

print(f"Prepared text chunk vectors: {len(text_chunk_records)}")

Book text files found: 12


Processing cleaned book text: 0it [00:00, ?it/s]

Prepared text chunk vectors: 0


## 6. Upsert Embeddings and Metadata into VectorDB

Build vector records and upsert in batches with idempotent IDs.

In [11]:
def chunked(seq, size: int):
    for i in range(0, len(seq), size):
        yield seq[i : i + size]


records = []
for _id, cap_obj in captions_state.items():
    emb_obj = embed_state.get(_id)
    if not emb_obj:
        continue

    records.append(
        {
            "id": _id,
            "embedding": emb_obj["embedding"],
            "document": cap_obj["caption"],
            "metadata": {
                "source_type": "image_caption",
                "relative_path": cap_obj["relative_path"],
                "vision_model": cap_obj["vision_model"],
                "embedding_model": emb_obj["embedding_model"],
                "captioned_at": cap_obj["captioned_at"],
                "embedded_at": emb_obj["embedded_at"],
                "indexed_at": datetime.now(timezone.utc).isoformat(),
            },
        }
    )

# Add book text chunks (prepared in Section 5.5)
if "text_chunk_records" in globals() and text_chunk_records:
    records.extend(text_chunk_records)

UPSERT_BATCH = 64
for batch in tqdm(list(chunked(records, UPSERT_BATCH)), desc="Upserting to Chroma"):
    collection.upsert(
        ids=[r["id"] for r in batch],
        embeddings=[r["embedding"] for r in batch],
        documents=[r["document"] for r in batch],
        metadatas=[r["metadata"] for r in batch],
    )

print("Total indexed:", collection.count())
print("Image caption records:", len([r for r in records if r["metadata"].get("source_type") == "image_caption"]))
print("Book text chunk records:", len([r for r in records if r["metadata"].get("source_type") == "book_text_chunk"]))

Upserting to Chroma:   0%|          | 0/16 [00:00<?, ?it/s]

Total indexed: 969
Image caption records: 969
Book text chunk records: 0


## 7. Run Similarity Search to Validate Stored Data

Generate a query embedding, run top-k search, and inspect matched image captions and paths.

In [12]:
def embed_query(query: str) -> List[float]:
    return embed_texts([query])[0]


sample_query = "What is Operating System ?"
query_vec = embed_query(sample_query)

search_res = collection.query(
    query_embeddings=[query_vec],
    n_results=8,
    include=["documents", "metadatas", "distances"],
)

for i, (doc, meta, dist) in enumerate(
    zip(
        search_res["documents"][0],
        search_res["metadatas"][0],
        search_res["distances"][0],
    ),
    start=1,
):
    src = meta.get("source_type", "unknown") if isinstance(meta, dict) else "unknown"
    locator = meta.get("relative_path") if isinstance(meta, dict) else ""
    if not locator and isinstance(meta, dict):
        locator = meta.get("book", "")
    print(f"#{i} | source={src} | distance={dist:.4f} | ref={locator}")
    print(f"text: {doc[:220]}\n")

2026-03-08 16:37:14,067 | INFO | HTTP Request: POST https://embedding-llm.ipropel.co.in/v1/embeddings "HTTP/1.1 200 OK"


#1 | source=image_caption | distance=0.2775 | ref=images/books/OS - 1_page3_img1.jpeg
text: This figure introduces the core concept of an operating system (OS), which serves as the intermediary between hardware and user applications. Key components include the kernel (managing CPU, memory, and I/O), process man

#2 | source=image_caption | distance=0.3950 | ref=images/books/OS - 1_page5_img2.jpeg
text: This figure introduces the foundational domain of Operating System Concepts, a core area in computer science focused on managing hardware resources and providing services to applications. Key components include process m

#3 | source=image_caption | distance=0.4807 | ref=images/books/OS - 2_page1134_img1.jpeg
text: This textbook covers the design and implementation of operating systems using MINIX 3, a microkernel-based educational OS. Key components include the kernel, process management, memory management, and device drivers, all

#4 | source=image_caption | distance=0.4809 | ref=image

## 8. Add Batching, Retry Logic, and Progress Logging

This cell wraps the core pipeline into a resumable runner that writes checkpoints and logs progress.

In [54]:
PIPELINE_STATUS = CHECKPOINT_DIR / "pipeline_status.json"


def save_status(stage: str, extra: Dict[str, Any] | None = None):
    payload = {
        "stage": stage,
        "time": datetime.now(timezone.utc).isoformat(),
        "captions": len(captions_state),
        "embeddings": len(embed_state),
        "indexed": collection.count(),
    }
    if extra:
        payload.update(extra)
    dump_json(PIPELINE_STATUS, payload)


def run_pipeline_resume_safe():
    save_status("start")
    logger.info("Resume-safe pipeline status written to %s", PIPELINE_STATUS)
    logger.info("Captions checkpoint: %s", CAPTION_CHECKPOINT)
    logger.info("Embeddings checkpoint: %s", EMBED_CHECKPOINT)
    logger.info("Current indexed count: %s", collection.count())
    save_status("done")


run_pipeline_resume_safe()

2026-03-08 16:28:43,337 | INFO | Resume-safe pipeline status written to /media/sasank-v/New Volume/Studies/College/Interships/LLM-Finetuning/notebooks/preprocessing/checkpoints/pipeline_status.json
2026-03-08 16:28:43,338 | INFO | Captions checkpoint: /media/sasank-v/New Volume/Studies/College/Interships/LLM-Finetuning/notebooks/preprocessing/checkpoints/captions.json
2026-03-08 16:28:43,338 | INFO | Embeddings checkpoint: /media/sasank-v/New Volume/Studies/College/Interships/LLM-Finetuning/notebooks/preprocessing/checkpoints/caption_embeddings.json
2026-03-08 16:28:43,339 | INFO | Current indexed count: 4203


## 9. Optional: Generate Synthetic Fine-Tuning Data from Books + Retrieved Context

This section reads `*_cleaned.txt` books, retrieves related image-caption context from the vector DB, and asks a larger text model to create instruction-output pairs saved as JSONL.

In [36]:
BOOK_GLOB = "*_cleaned.txt"
SYNTH_OUT = OUTPUT_DIR / "synthetic_finetune.jsonl"


def load_books(books_dir: Path) -> List[Tuple[str, str]]:
    items = []
    for p in sorted(books_dir.glob(BOOK_GLOB)):
        txt = p.read_text(encoding="utf-8", errors="ignore")
        items.append((p.name, txt))
    return items


def split_text(text: str, chunk_chars: int = 1800, overlap: int = 250) -> List[str]:
    chunks = []
    start = 0
    while start < len(text):
        end = min(len(text), start + chunk_chars)
        chunks.append(text[start:end])
        if end == len(text):
            break
        start = end - overlap
    return chunks


def retrieve_caption_context(topic: str, top_k: int = 3) -> str:
    vec = embed_query(topic)
    res = collection.query(
        query_embeddings=[vec],
        n_results=top_k,
        where={"source_type": "image_caption"},
        include=["documents", "metadatas", "distances"],
    )
    lines = []
    for d, m, s in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        lines.append(f"- ({s:.4f}) {m.get('relative_path')}: {d}")
    return "\n".join(lines)


def generate_synthetic_pairs(topic: str, context_chunk: str, image_ctx: str, n_pairs: int = 3) -> List[Dict[str, str]]:
    prompt = f"""
You are generating high-quality fine-tuning data for a CS tutoring model.
Create exactly {n_pairs} JSON objects in a JSON array.
Each object must contain keys: instruction, input, output, topic, source.

Topic: {topic}
Book Context:\n{context_chunk}
Image Retrieval Context:\n{image_ctx}

Requirements:
- Keep outputs factual, concise, and educational.
- Include a mix of conceptual and problem-solving style instructions.
- source must be 'synthetic'.
Return only valid JSON.
""".strip()

    def _call():
        resp = text_client.chat.completions.create(
            model=SYNTHETIC_MODEL,
            temperature=0.7,
            messages=[
                {"role": "system", "content": "You produce strict JSON outputs."},
                {"role": "user", "content": prompt},
            ],
        )
        return (resp.choices[0].message.content or "[]").strip()

    raw = with_retry(_call)
    try:
        data = json.loads(raw)
        if isinstance(data, list):
            return data
    except Exception:
        logger.warning("Synthetic JSON parse failed for topic=%s", topic)
    return []


books = load_books(BOOKS_DIR)
print(f"Loaded {len(books)} cleaned books")

all_pairs: List[Dict[str, str]] = []
MAX_CHUNKS_PER_BOOK = 8

for book_name, text in tqdm(books, desc="Generating synthetic data"):
    chunks = split_text(text)
    topic = book_name.replace("._cleaned.txt", "").strip()
    image_ctx = retrieve_caption_context(topic, top_k=3) if collection.count() > 0 else ""

    for c in chunks[:MAX_CHUNKS_PER_BOOK]:
        pairs = generate_synthetic_pairs(topic=topic, context_chunk=c, image_ctx=image_ctx, n_pairs=3)
        for p in pairs:
            p.setdefault("topic", topic)
            p.setdefault("source", "synthetic")
            p["book"] = book_name
            all_pairs.append(p)

with SYNTH_OUT.open("w", encoding="utf-8") as f:
    for row in all_pairs:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"Saved {len(all_pairs)} records to {SYNTH_OUT}")

Loaded 2 cleaned books


Generating synthetic data:   0%|          | 0/2 [00:00<?, ?it/s]

2026-03-08 15:39:59,010 | INFO | HTTP Request: POST https://embedding-llm.ipropel.co.in/v1/embeddings "HTTP/1.1 200 OK"
2026-03-08 15:40:02,923 | INFO | HTTP Request: POST https://llm.ipropel.co.in/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-08 15:40:06,097 | INFO | HTTP Request: POST https://llm.ipropel.co.in/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-08 15:40:09,989 | INFO | HTTP Request: POST https://llm.ipropel.co.in/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-08 15:40:09,992 | WARNING | Synthetic JSON parse failed for topic=CAO - 1
2026-03-08 15:40:13,676 | INFO | HTTP Request: POST https://llm.ipropel.co.in/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-08 15:40:17,874 | INFO | HTTP Request: POST https://llm.ipropel.co.in/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-08 15:40:20,435 | INFO | HTTP Request: POST https://llm.ipropel.co.in/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-08 15:40:23,405 | INFO | HTTP Request: POST https://llm.ipropel.co.in/v1/chat/completions "HTT

Saved 45 records to /media/sasank-v/New Volume/Studies/College/Interships/LLM-Finetuning/notebooks/preprocessing/outputs/synthetic_finetune.jsonl
